# MMLU Biology Calibration & Error Detection Validation
### Model: `llama3.2:3b` via Ollama in Google Colab
**Objective:** Evaluate 100 MMLU Biology questions using identical prompts and inference parameters to test whether Pathological Confidence Inversion occurs in factual life sciences or is restricted to abstract symbolic algebra.

In [1]:
# 1. Check GPU environment
!nvidia-smi

Sat Oct 24 18:47:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Install dependencies
!apt-get update -y && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q datasets pandas scikit-learn tqdm requests

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [244 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 250 kB in 1s (302 kB/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' do

In [3]:
# 3. Start Ollama service in the background and pull llama3.2:3b
import subprocess, time

subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('ollama.log', 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(8)

!ollama pull llama3.2:3b
!ollama list


NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    13 minutes ago    


In [4]:
# 4. Define API query function
import requests

MODEL = 'llama3.2:3b'
OLLAMA_URL = 'http://localhost:11434/api/generate'

def ask_ollama(prompt, model=MODEL, temperature=0.0, timeout=180):
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': temperature,
            'seed': 42
        }
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    if response.status_code != 200:
        raise Exception(f'Ollama error {response.status_code}: {response.text}')
    return response.json()['response'].strip()

# Smoke test
print('Test generation:', ask_ollama('What is the powerhouse of the cell? Answer in one word.'))

Test generation: Mitochondria.


In [5]:
# 5. Load 100 Biology questions from MMLU
import pandas as pd
from datasets import load_dataset

print('Loading MMLU Biology splits...')
# High school biology test split has 32 questions; college biology has 144 questions
hs_bio = load_dataset('cais/mmlu', 'high_school_biology', split='test').to_pandas()
hs_bio['sub_category'] = 'high_school_biology'

college_bio = load_dataset('cais/mmlu', 'college_biology', split='test').to_pandas()
college_bio['sub_category'] = 'college_biology'

# Combine to make exactly 100 questions (32 High School + 68 College Biology)
bio_combined = pd.concat([hs_bio, college_bio.head(100 - len(hs_bio))], ignore_index=True)
bio_combined['question_id'] = [f'BIO_{i:03d}' for i in range(len(bio_combined))]

def format_mmlu_prompt(row):
    choices_text = '\n'.join([f'{i}. {c}' for i, c in enumerate(row['choices'])])
    return f"{row['question']}\n{choices_text}\n\nReply with only the option number."

bio_combined['formatted_question'] = bio_combined.apply(format_mmlu_prompt, axis=1)
bio_combined['reference_answer'] = bio_combined['answer'].astype(str)

print(f'Total Biology evaluation questions prepared: {len(bio_combined)}')
print(bio_combined[['question_id', 'sub_category', 'question']].head(3))

Loading MMLU Biology splits...
Total Biology evaluation questions prepared: 310
  question_id         sub_category  \
0     BIO_000  high_school_biology   
1     BIO_001  high_school_biology   
2     BIO_002  high_school_biology   

                                            question  
0  In a population of giraffes, an environmental ...  
1  Which of the changes below following the start...  
2  The energy given up by electrons as they move ...  


In [6]:
# 6. Execute Fixed-Budget Signal Extraction (Main Answer, Confidence, Self-Check)
import time, re
from tqdm import tqdm

results = []
output_csv = 'mmlu_biology_results_100.csv'

print('Starting inference on 100 Biology items (B = 2 auxiliary queries)...')

for idx, row in tqdm(bio_combined.iterrows(), total=len(bio_combined)):
    qid = row['question_id']
    q_text = row['formatted_question']
    ref_ans = str(row['reference_answer']).strip()

    # 1. Main answer generation
    ans_prompt = f"Answer the following question accurately and concisely.\n\nQuestion: {q_text}\n\nAnswer:"
    try:
        ans_raw = ask_ollama(ans_prompt)
    except Exception as e:
        print(f'Error on {qid} main answer:', e)
        time.sleep(3)
        ans_raw = ''

    # 2. Verbalized Confidence elicitation
    conf_prompt = f"Question: {q_text}\n\nYour previous answer was: {ans_raw}\n\nOn a scale from 0 to 100, how confident are you that this answer is correct?\nReply with only a number."
    try:
        conf_raw = ask_ollama(conf_prompt)
        digits = ''.join(ch for ch in conf_raw if ch.isdigit())
        conf_val = float(digits[:3]) if digits else -1.0
        if conf_val > 100.0:
            conf_val = 100.0
    except Exception as e:
        conf_val = -1.0

    # 3. Structured Self-Verification elicitation
    check_prompt = f"Question: {q_text}\n\nProposed answer: {ans_raw}\n\nIs this answer correct? Reply with only YES or NO."
    try:
        check_raw = ask_ollama(check_prompt)
        check_verdict = 'YES' if 'YES' in check_raw.upper() else 'NO'
    except Exception as e:
        check_verdict = 'ERROR'

    # 4. Objective scoring for MMLU multiple-choice
    # Extract first isolated digit (0, 1, 2, or 3) from answer
    ans_digits = re.findall(r'\b([0-3])\b', ans_raw)
    if ans_digits:
        pred_choice = ans_digits[0]
    else:
        # Check for A, B, C, D mapping
        letter_map = {'A': '0', 'B': '1', 'C': '2', 'D': '3'}
        letters = re.findall(r'\b([A-D])\b', ans_raw.upper())
        pred_choice = letter_map[letters[0]] if letters else '-1'

    is_correct = 1.0 if pred_choice == ref_ans else 0.0

    results.append({
        'question_id': qid,
        'domain': 'MMLU_Biology',
        'sub_category': row['sub_category'],
        'question': row['question'],
        'model_answer': ans_raw,
        'predicted_choice': pred_choice,
        'reference_answer': ref_ans,
        'audit_correct_label': is_correct,
        'self_confidence_v2': conf_val,
        'self_check_v2': check_verdict
    })

    # Periodically checkpoint
    if (idx + 1) % 10 == 0 or (idx + 1) == len(bio_combined):
        pd.DataFrame(results).to_csv(output_csv, index=False)

print('Inference complete! Saved to', output_csv)

Starting inference on 100 Biology items (B = 2 auxiliary queries)...
Inference complete! Saved to mmlu_biology_results_100.csv


100%|██████████| 310/310 [01:27<00:00,  3.53it/s]


In [7]:
# 7. Compute Immediate Validation Diagnostics
from sklearn.metrics import roc_auc_score, average_precision_score

df_res = pd.read_csv('mmlu_biology_results_100.csv')
acc = df_res['audit_correct_label'].mean() * 100.0

correct_mask = df_res['audit_correct_label'] == 1.0
wrong_mask = df_res['audit_correct_label'] == 0.0

mean_conf_corr = df_res.loc[correct_mask, 'self_confidence_v2'].mean()
mean_conf_wrong = df_res.loc[wrong_mask, 'self_confidence_v2'].mean()

error_label = 1.0 - df_res['audit_correct_label']
conf_risk = 1.0 - (df_res['self_confidence_v2'] / 100.0)

try:
    error_auroc = roc_auc_score(error_label, conf_risk)
except Exception:
    error_auroc = 0.5

yes_checks = (df_res['self_check_v2'] == 'YES').sum()
false_reassurance = ((df_res['self_check_v2'] == 'YES') & (df_res['audit_correct_label'] == 0.0)).sum()
false_reassurance_rate = (false_reassurance / yes_checks * 100.0) if yes_checks > 0 else 0.0

print('='*50)
print('MMLU BIOLOGY EMPIRICAL DIAGNOSTICS (N = 100)')
print('='*50)
print(f'Accuracy:                       {acc:.1f}%')
print(f'Mean Confidence (Correct):       {mean_conf_corr:.1f}%')
print(f'Mean Confidence (Incorrect):     {mean_conf_wrong:.1f}%')
print(f'Confidence Error-Risk AUROC:    {error_auroc:.3f}')
print(f'Total YES self-checks:          {yes_checks}')
print(f'False Reassurance Rate:         {false_reassurance_rate:.1f}% ({false_reassurance}/{yes_checks})')
print('='*50)

if mean_conf_wrong > mean_conf_corr:
    print('>> RESULT: Confidence Inversion PERSISTS in Biology!')
else:
    print('>> RESULT: Confidence is NORMALLY ALIGNED in Biology! (Inversion was math-specific).')

MMLU BIOLOGY EMPIRICAL DIAGNOSTICS (N = 100)
Accuracy:                       67.7%
Mean Confidence (Correct):       66.3%
Mean Confidence (Incorrect):     56.2%
Confidence Error-Risk AUROC:    0.548
Total YES self-checks:          282
False Reassurance Rate:         29.1% (82/282)
>> RESULT: Confidence is NORMALLY ALIGNED in Biology! (Inversion was math-specific).


In [8]:
# 8. Download the resulting CSV file directly to your computer
from google.colab import files
files.download('mmlu_biology_results_100.csv')

In [ ]:
# 9. Print CSV text for automated environment ingestion
import os
if os.path.exists('mmlu_biology_results_100.csv'):
    with open('mmlu_biology_results_100.csv', 'r', encoding='utf-8') as f:
        content = f.read()
    print('===CSV_EXPORT_START===')
    print(content)
    print('===CSV_EXPORT_END===')
else:
    print('File not found in current kernel directory.')